# Sarashina 2.2 Export

Prepare SB Intuitions Sarashina 2.2 models for use with the Lenzu runtime.

**Scope:**
- `sarashina2.2-0.5b-instruct-v0.1` — exported to ONNX (text-only, runs under `ort`).
- `sarashina2.2-ocr`, `sarashina2.2-vision-3b` — pre-cached to Drive, run via the HuggingFace `transformers` Python runtime as SB Intuitions designed them. ONNX export is not viable for these custom `sarashina2_vision` architectures.

**Runtime:** Google Colab (T4 / A100 GPU recommended)

<a href="https://colab.research.google.com/github/HidekiAI/lenzu/blob/trunk/notebooks/sarashina_export.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install with the rust-accelerated transfer layer
!pip install -U "optimum[onnxruntime-gpu]" transformers accelerate hf_transfer
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

: 

In [ ]:
from google.colab import drive
import os

# 1. Mount your 20TB Drive
drive.mount('/content/drive')

# 2. Point the Hugging Face cache to your Drive
# This way, the 8GB is saved PERMANENTLY in your 20TB pool
os.environ["HF_HOME"] = "/content/drive/MyDrive/HF_Cache"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

from huggingface_hub import snapshot_download

models = ["sbintuitions/sarashina2.2-ocr", "sbintuitions/sarashina2.2-vision-3b"]
for m in models:
    print(f"Downloading {m} directly to Google Drive...")
    snapshot_download(repo_id=m, cache_dir="/content/drive/MyDrive/HF_Cache")
    print(f"{m} is now safely stored in your Drive!")

In [ ]:
import os
import shutil
import subprocess
import torch
from google.colab import drive

drive.mount('/content/drive', force_remount=True)
DRIVE_PATH = "/content/drive/MyDrive/Lenzu_Exports"
os.makedirs(DRIVE_PATH, exist_ok=True)

# Only the 0.5b instruct model is exported to ONNX. The vision models
# (sarashina2.2-ocr, sarashina2.2-vision-3b) were designed by SB Intuitions
# for the HuggingFace transformers Python runtime, not ONNX.
MODEL_ID = "sbintuitions/sarashina2.2-0.5b-instruct-v0.1"
MODEL_NAME = "mini_500m"
TASK = "text-generation-with-past"

# fp16 only works on CUDA; fall back to fp32/CPU if GPU is unavailable
if torch.cuda.is_available():
    DEVICE, DTYPE = "cuda", "fp16"
else:
    DEVICE, DTYPE = "cpu", "fp32"
    print("WARNING: no CUDA -- falling back to CPU/fp32 (slower, larger file)")

out_dir = f"./{MODEL_NAME}_onnx"
zip_file = f"{MODEL_NAME}_export.zip"

if os.path.exists(f"{DRIVE_PATH}/{zip_file}"):
    print(f"Skipping {MODEL_NAME}, already in Drive.")
else:
    print(f"Forging {MODEL_NAME} with task {TASK} on {DEVICE}/{DTYPE}...")
    cmd = (
        f'optimum-cli export onnx'
        f' --model "{MODEL_ID}"'
        f' --task "{TASK}"'
        f' --trust-remote-code'
        f' --device {DEVICE}'
        f' --dtype {DTYPE}'
        f' "{out_dir}"'
    )
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"STDOUT: {result.stdout[-2000:]}")
        print(f"STDERR: {result.stderr[-2000:]}")
        raise RuntimeError(f"Export failed with exit code {result.returncode}")

    shutil.make_archive(MODEL_NAME, 'zip', out_dir)
    shutil.move(f"{MODEL_NAME}.zip", f"{DRIVE_PATH}/{zip_file}")
    shutil.rmtree(out_dir)
    print(f"{MODEL_NAME} SUCCESS.")